In [5]:
import torch

from model.FeedForward import FeedForwardNetwork
from model.attention import scaled_dot_product_attention, MultiHeadAttention
from model.encoder import Encoder

torch.manual_seed(42)
#3 token,each with 4 features
Q = torch.randn(1,3,4)
K = torch.randn(1,3,4)
V = torch.randn(1,3,4)
output,attention = scaled_dot_product_attention(Q,K,V)

print("Output:", output.shape)
print("Attention:",attention.shape)
print("Attention weights:\n",attention)
print("Output:\n",output)
print("Row sums:", attention.sum(dim=-1))


Output: torch.Size([1, 3, 4])
Attention: torch.Size([1, 3, 3])
Attention weights:
 tensor([[[0.3017, 0.3098, 0.3884],
         [0.2451, 0.3801, 0.3748],
         [0.2938, 0.2293, 0.4769]]])
Output:
 tensor([[[ 0.1212,  0.5156, -0.2394, -0.1912],
         [ 0.0999,  0.5376, -0.2558, -0.1143],
         [ 0.1348,  0.5492, -0.3327, -0.3267]]])
Row sums: tensor([[1.0000, 1.0000, 1.0000]])


In [6]:
model = MultiHeadAttention(d_model=512,num_heads=8)
# Batch size = 2, sequence length = 4, features = 512
X = torch.randn(2,4,512)
output, attention = model(X,X,X,mask=None)

print("Output shape:", output.shape)
print("Attention shape:", attention.shape)
print("Attention row sums:", attention.sum(dim=-1))

Output shape: torch.Size([2, 4, 512])
Attention shape: torch.Size([2, 8, 4, 4])
Attention row sums: tensor([[[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)


In [7]:
model = FeedForwardNetwork(d_model=512,d_ff=2048)
X = torch.randn(2,4,512)
output = model(X)
print("Input shape:", X.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 512])
Output shape: torch.Size([2, 4, 512])


In [8]:

encoder = Encoder(
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    dropout=0.1
)

x = torch.randn(2, 4, 512)

output = encoder(x)

print("Input shape:", x.shape)
print("Encoder output shape:", output.shape)
print("Number of encoder layers:", len(encoder.layers))

TypeError: Encoder.__init__() missing 1 required positional argument: 'vocab_size'

In [ ]:
from model.embedding import InputEmbedding
embedding = InputEmbedding(vocab_size=10000, d_model=512)
tokens = torch.tensor([
    [15, 87, 204, 319],
    [21, 45, 102, 300]
])
output = embedding(tokens)
print("Input shape:", tokens.shape)
print("Output shape:", output.shape)
print("Embedding matrix:", embedding.embedding.weight.shape)

In [ ]:
from model.positional_encoding import PositionalEncoding
pos_encoding = PositionalEncoding(d_model=512, max_len=100)

x = torch.zeros(2, 4, 512)

output = pos_encoding(x)

print("Output shape:", output.shape)
print("Position 0:", output[0, 0, :6])
print("Position 1:", output[0, 1, :6])

In [ ]:
from model.encoder import Encoder
encoder = Encoder(
    vocab_size=10000,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6,
    dropout=0.1
)
# Two sentences, each containing four token IDs
tokens = torch.tensor([
    [15, 87, 204, 319],
    [21, 45, 102, 300]

])

output = encoder(tokens)
print("Input shape:", tokens.shape)
print("Encoder output shape:", output.shape)

In [ ]:

from model.decoder import DecoderLayer

decoder_layer = DecoderLayer(
    d_model=512,
    num_heads=8,
    d_ff=2048
)

y = torch.randn(2, 3, 512)
encoder_output = torch.randn(2, 4, 512)

causal_mask = torch.triu(
    torch.ones(3, 3, dtype=torch.bool),
    diagonal=1
)

output = decoder_layer(
    y,
    encoder_output,
    causal_mask=causal_mask
)

print("Decoder output shape:", output.shape)

In [9]:
from model.decoder import Decoder

decoder = Decoder(
    vocab_size=10000,
    d_model=512,
    num_heads=8,
    d_ff=2048,
    num_layers=6

)

# Encoder output: 2 sentences, 4 source tokens eac
encoder_output = torch.randn(2, 4, 512)
# Decoder input: 2 sentences, 3 target tokens each
target_tokens = torch.tensor([
    [1, 45, 87],
    [1, 62, 103]
])

# Causal mask: prevent attending to future tokens
causal_mask = torch.triu(
    torch.ones(3, 3, dtype=torch.bool),
    diagonal=1

)
output = decoder(
    target_tokens,
    encoder_output,
    causal_mask=causal_mask
)

print("Decoder input shape:", target_tokens.shape)
print("Decoder output shape:", output.shape)
print("Number of decoder layers:", len(decoder.layers))

Decoder input shape: torch.Size([2, 3])
Decoder output shape: torch.Size([2, 3, 512])
Number of decoder layers: 6
